# JSON y Datos Semiestructurados

# JSON anidado: structs, arrays y el tipo VARIANT

Los datos reales rara vez llegan como tablas planas. Las APIs, los eventos de apps y los
logs llegan como **JSON anidado**: objetos dentro de objetos, listas de valores. En este
laboratorio los eventos de ratings vienen así:

```json
{"event_id": 1, "user": {"id": 828, "country": "CR", "plan": "premium"},
 "movie": {"id": 318, "genres": ["Drama", "Sci-Fi"]}, "rating": 4.5, "ts": "..."}
```

Tres estrategias en Databricks:

1. **Aplanar a STRUCT/columnas** al ingerir — rápido de consultar, pero rígido: cualquier
   cambio del productor rompe el esquema.
2. **Guardar el JSON como string** y parsear al leer — flexible, pero lento: cada consulta
   re-parsea todo el texto.
3. **Tipo `VARIANT`** (DBR 15.3+) — binario autodescriptivo: flexible ante cambios de
   esquema **y** mucho más rápido que strings JSON. Es la recomendación actual.

Veremos primero la exploración clásica (structs, `explode`) y luego la ingesta VARIANT.

In [ ]:
%run ./70_Setup_Landing_Zone

In [ ]:
# Limpieza local idempotente: este notebook resetea SU propio estado.
dbutils.fs.rm(LANDING_JSON, recurse=True)
dbutils.fs.mkdirs(LANDING_JSON)
dbutils.fs.rm(CHK_43, recurse=True)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.eventos_variant")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.eventos_struct")   # por si quedó de otra corrida
print("Estado del notebook 43 reiniciado.")

In [ ]:
# Lote 1 de eventos ANIDADOS (300 eventos)
generar_lote_ratings(LANDING_JSON, lote=1, n=300, anidado=True)
display(dbutils.fs.ls(LANDING_JSON))

> **Antes de ejecutar:** El campo `movie.genres` es una lista con 1–3 géneros por evento. Si leemos este JSON con `spark.read.json`, ¿qué tipo de dato tendrá la columna `movie`: string, struct o array? ¿Y `movie.genres`?

In [ ]:
# Exploración batch: dejar que Spark infiera la estructura anidada
df_nested = spark.read.json(f"{LANDING_JSON}/lote_01.json")
df_nested.printSchema()
display(df_nested.limit(5))

`movie` es un **struct** (objeto con campos) y `movie.genres` es un **array<string>**.
Para trabajar con ellos:

* **Notación de punto** para campos anidados: `col("user.country")`.
* **`explode`** para convertir cada elemento de un array en una fila propia —
  el mismo patrón `from_json` / `explode` que vimos en la Clase 6.

In [ ]:
from pyspark.sql.functions import col, explode, round as sround, avg, count

# Promedio de rating por género: explode convierte cada género en una fila
por_genero = (df_nested
              .select(col("user.country").alias("country"),
                      explode(col("movie.genres")).alias("genero"),
                      col("rating"))
              .groupBy("genero")
              .agg(count("*").alias("eventos"),
                   sround(avg("rating"), 2).alias("rating_promedio"))
              .orderBy(col("rating_promedio").desc()))

display(por_genero)

## VARIANT: flexible Y rápido

El dilema clásico: el string JSON es flexible pero lento; el struct es rápido pero rígido.
`VARIANT` resuelve ambos lados — almacena el dato en un binario autodescriptivo que no
exige esquema fijo y se consulta mucho más rápido que parsear strings.

Con Auto Loader se puede ingerir **todo el registro** como una sola columna VARIANT usando
la opción `singleVariantColumn`. En ese modo **no hay `_rescued_data` ni evolución de
esquema — porque no hacen falta**: no hay esquema rígido contra el cual chocar.

Si mañana los eventos traen un campo nuevo `user.city`, ¿la ingesta VARIANT fallaría como falló Auto Loader con `addNewColumns` en el notebook 41, o seguiría funcionando sin cambios?

In [ ]:
def ingesta_variant():
    stream = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "json")
              .option("singleVariantColumn", "evento")   # TODO el registro -> una columna VARIANT
              .option("cloudFiles.schemaLocation", f"{CHK_43}/schema")
              .load(LANDING_JSON)
              .writeStream
              .option("checkpointLocation", f"{CHK_43}/chk")
              .trigger(availableNow=True)
              .toTable(f"{CATALOG}.{SCHEMA}.eventos_variant"))
    stream.awaitTermination()

# Ingesta del lote 1
ingesta_variant()

# Lote 2: trae ADEMÁS el campo nuevo user.city
generar_lote_ratings(LANDING_JSON, lote=2, n=300, anidado=True, columnas_extra=True)

# La MISMA ingesta, sin cambios, sin try/except:
ingesta_variant()

tabla = spark.table(f"{CATALOG}.{SCHEMA}.eventos_variant")
tabla.printSchema()   # una sola columna: evento (VARIANT)
print("Filas totales:", tabla.count())

**No falló.** El lote 2 con `user.city` entró sin reinicio ni excepción — el VARIANT
absorbe el campo nuevo. Compara con el `try/except` que necesitamos en el notebook 41.

## Consultar VARIANT: la sintaxis de dos puntos

* `:` navega campos — `evento:user:country`
* `[n]` indexa arrays — `evento:movie:genres[0]`
* `::tipo` castea — `evento:rating::double`
* Un campo **ausente** devuelve `NULL` (no falla).

In [ ]:
%sql
SELECT evento:user:country::string    AS country,
       evento:rating::double          AS rating,
       evento:movie:genres[0]::string AS primer_genero,
       evento:user:city::string       AS city
FROM   workspace.ingesta_demo.eventos_variant
LIMIT 10;

El campo `user.city` solo existe en el lote 2. Al agrupar por `city`, ¿qué grupo va a concentrar exactamente 300 filas?

In [ ]:
%sql
-- city es NULL en las 300 filas del lote 1 (campo ausente -> NULL, sin fallar)
-- y tiene valor en las 300 del lote 2: evidencia de la flexibilidad del VARIANT
SELECT evento:user:city::string AS city,
       COUNT(*)                 AS eventos
FROM   workspace.ingesta_demo.eventos_variant
GROUP  BY city
ORDER  BY eventos DESC;

In [ ]:
%sql
-- Agregado analítico directamente sobre el VARIANT: rating promedio por país
SELECT evento:user:country::string      AS country,
       ROUND(AVG(evento:rating::double), 2) AS rating_promedio,
       COUNT(*)                          AS eventos
FROM   workspace.ingesta_demo.eventos_variant
GROUP  BY country
ORDER  BY rating_promedio DESC;

## Buenas prácticas de cierre

* **Patrón Bronze VARIANT a Silver tipado:** los campos que se consultan con frecuencia se
  extraen a columnas normales en Silver (`evento:user:country::string AS country`, ...).
  Es la conexión directa con la arquitectura Medallion de la clase pasada.
* **Limitaciones de VARIANT:** no sirve como clave de clustering ni de particionado, y no
  se puede comparar ni usar en `GROUP BY` directamente — primero hay que extraer y castear
  el campo.